<a href="https://colab.research.google.com/github/Mogi21/Master-Thesis-Mammography-VLM/blob/main/Faster_RCNN_Baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# PHASE 1: FASTER R-CNN BASELINE PIPELINE (VINDR-MAMMO)
# ==============================================================================

# 1. Environment Setup
!pip install -q pydicom pylibjpeg pylibjpeg-libjpeg pylibjpeg-openjpeg
import os
import zipfile
import shutil
import pandas as pd
import pydicom
import numpy as np
import cv2
import torch
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# ==============================================================================
# 2. Data Preparation & Conversion
# ==============================================================================
base_drive_dir = '/content/drive/MyDrive/vindr_raw_data_11'
zip_path = '/content/drive/MyDrive/vindr_pngs_11'
if not os.path.exists(zip_path):
    zip_path = '/content/drive/MyDrive/vindr_pngs_11.zip'

new_dicom_drive_folder = f'{base_drive_dir}/Missing_DICOMs'
local_dir = '/content/dataset/images'
csv_path = f'{base_drive_dir}/finding_annotations.csv'
drive_csv_path = f'{base_drive_dir}/baseline_annotations.csv'

os.makedirs(local_dir, exist_ok=True)

print("Extracting original archive of PNGs to local GPU storage...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(local_dir)

print(f"\nConverting remaining DICOM images from {new_dicom_drive_folder} to PNG...")
new_copied_count = 0
if os.path.exists(new_dicom_drive_folder):
    for root, dirs, files in os.walk(new_dicom_drive_folder):
        for filename in files:
            if filename.lower().endswith(('.dicom', '.dcm')):
                dicom_path = os.path.join(root, filename)
                png_filename = filename.lower().replace('.dicom', '.png').replace('.dcm', '.png')
                out_path = os.path.join(local_dir, png_filename)

                if not os.path.exists(out_path):
                    try:
                        ds = pydicom.dcmread(dicom_path)
                        image = ds.pixel_array.astype(float)
                        image = (np.maximum(image, 0) / image.max()) * 255.0
                        image = np.uint8(image)
                        cv2.imwrite(out_path, image)
                        new_copied_count += 1
                    except Exception:
                        pass
print(f"Successfully converted {new_copied_count} new DICOM images to PNG!")

extracted_files = [f for root, dirs, files in os.walk(local_dir) for f in files if f.lower().endswith('.png')]
available_pngs = set([f.replace('.png', '').replace('.PNG', '') for f in extracted_files])
print(f"Successfully loaded a total of {len(available_pngs)} images for training.")

print("Regenerating baseline annotations for all images...")
df = pd.read_csv(csv_path)
filtered_df = df[df['image_id'].isin(available_pngs)]
filtered_df.to_csv(drive_csv_path, index=False)

print("Backing up the complete dataset of PNGs to Google Drive...")
shutil.make_archive('/content/vindr_pngs_full', 'zip', '/content/dataset/images')
shutil.move('/content/vindr_pngs_full.zip', '/content/drive/MyDrive/vindr_pngs_full.zip')
print("Backup complete!")

# ==============================================================================
# 3. Dataset & DataLoader Setup
# ==============================================================================
class VinDrDataset(Dataset):
    def __init__(self, csv_file, img_dir):
        self.img_dir = img_dir
        self.df = pd.read_csv(csv_file)
        self.df = self.df.dropna(subset=['xmin', 'ymin', 'xmax', 'ymax'])
        self.image_ids = self.df['image_id'].unique()

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        img_path = os.path.join(self.img_dir, f"{img_id}.png")
        image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        image = image / 255.0
        image = torch.as_tensor(image, dtype=torch.float32).unsqueeze(0)

        boxes = self.df[self.df['image_id'] == img_id][['xmin', 'ymin', 'xmax', 'ymax']].values
        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.ones((len(boxes),), dtype=torch.int64)

        target = {"boxes": boxes, "labels": labels, "image_id": torch.tensor([idx])}
        return image, target

def collate_fn(batch):
    return tuple(zip(*batch))

dataset = VinDrDataset(drive_csv_path, local_dir)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, collate_fn=collate_fn)

# ==============================================================================
# 4. Model Initialization & Training
# ==============================================================================
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model = torchvision.models.detection.fasterrcnn_resnet50_fpn(pretrained=True)
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, 2) # 2 classes: background + lesion
model.to(device)

optimizer = torch.optim.SGD(model.parameters(), lr=0.005, momentum=0.9, weight_decay=0.0005)
num_epochs = 2 # Adjusted for compute constraints

print(f"Starting Training on {device} for {num_epochs} epochs...")
epoch_losses = []
for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0
    for images, targets in train_loader:
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()
        epoch_loss += losses.item()

    avg_loss = epoch_loss / len(train_loader)
    epoch_losses.append(avg_loss)
    print(f"Epoch {epoch+1}/{num_epochs} - Loss: {avg_loss:.4f}")

torch.save(model.state_dict(), '/content/drive/MyDrive/faster_rcnn_vindr.pth')
print("Model saved to Drive!")

# ==============================================================================
# 5. Visualizations for Poster
# ==============================================================================
# 5.1 Training Loss Curve
epochs = list(range(1, num_epochs + 1))
plt.figure(figsize=(8, 5))
plt.plot(epochs, epoch_losses, marker='o', linestyle='-', color='#1f77b4', linewidth=2)
plt.title('Faster R-CNN Training Loss', fontsize=16, fontweight='bold')
plt.xlabel('Epoch', fontsize=14)
plt.ylabel('Average Loss', fontsize=14)
plt.xticks(epochs)
plt.grid(True, linestyle='--', alpha=0.7)
plt.savefig('/content/loss_curve_poster.png', dpi=300, bbox_inches='tight')
plt.close()

# 5.2 Detection Example
model.eval()
images, targets = next(iter(val_loader))
img_tensor = images[0].to(device)
gt_boxes = targets[0]['boxes'].cpu().numpy()

with torch.no_grad():
    predictions = model([img_tensor])

pred_boxes = predictions[0]['boxes'].cpu().numpy()
pred_scores = predictions[0]['scores'].cpu().numpy()
img_np = img_tensor.cpu().permute(1, 2, 0).numpy()

fig, ax = plt.subplots(1, 1, figsize=(8, 8))
ax.imshow(img_np, cmap='gray')

for box in gt_boxes:
    xmin, ymin, xmax, ymax = box
    ax.add_patch(patches.Rectangle((xmin, ymin), xmax-xmin, ymax-ymin,
                 linewidth=2.5, edgecolor='lime', facecolor='none', label='Ground Truth'))

for box, score in zip(pred_boxes, pred_scores):
    if score > 0.50:
        xmin, ymin, xmax, ymax = box
        ax.add_patch(patches.Rectangle((xmin, ymin), xmax-xmin, ymax-ymin,
                     linewidth=2.5, edgecolor='red', facecolor='none', label=f'Prediction ({score:.2f})'))

handles, labels = plt.gca().get_legend_handles_labels()
by_label = dict(zip(labels, handles))
ax.legend(by_label.values(), by_label.keys(), loc='upper right')
plt.axis('off')
plt.title('Detection Example', fontsize=14, fontweight='bold')
plt.savefig('/content/detection_example_poster.png', dpi=300, bbox_inches='tight')
plt.close()
print("Visualizations saved to /content/")